In [29]:
import pyspark
import pyspark.sql.functions as F
from pyspark.sql.functions import col

spark = pyspark.sql.SparkSession.builder \
    .appName("gold_layer_checks") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [39]:
gold_feature_df = spark.read.parquet("datamart/gold/feature_store/*")

display(gold_feature_df.toPandas().head())

,Customer_ID,snapshot_date,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,...,fe_14,fe_15,fe_16,fe_17,fe_18,fe_19,fe_20,debt_to_income_ratio,emi_to_salary_ratio,high_credit_utilization_flag
0,CUS_0x47a5,2023-01-01,Alexa,32.0,827-59-6035,Teacher,17680.689453,1449.390869,6.0,6.0,...,169.0,104.0,63.0,59.0,343.0,360.0,8.0,0.050223,0.021863,0
1,CUS_0x51c9,2023-01-01,Dinab,29.0,640-03-4819,Lawyer,105565.710938,8813.142578,7.0,3.0,...,5.0,55.0,86.0,261.0,-50.0,13.0,222.0,0.009708,0.000000,0
2,CUS_0xb0c4,2023-01-01,Pawelb,14.0,810-15-1395,Entrepreneur,9001.344727,811.112061,9.0,6.0,...,-59.0,123.0,84.0,6.0,31.0,-48.0,5.0,0.165826,0.028202,0
3,CUS_0xc63b,2023-01-01,John Acherd,28.0,563-40-6433,Media_Manager,32928.359375,3006.030029,5.0,6.0,...,100.0,-61.0,163.0,255.0,78.0,142.0,12.0,0.023721,0.000000,0
4,CUS_0x4c71,2023-01-01,Lucye,55.0,267-92-2329,Entrepreneur,32227.769531,2494.647461,3.0,6.0,...,-124.0,-51.0,76.0,110.0,144.0,74.0,157.0,0.013355,0.015208,0


In [40]:
gold_label_df = spark.read.parquet("datamart/gold/label_store/*")

display(gold_label_df.toPandas().head())

,loan_id,Customer_ID,label,label_def,snapshot_date
0,CUS_0x1037_2023_01_01,CUS_0x1037,0,30dpd_6mob,2023-07-01
1,CUS_0x1069_2023_01_01,CUS_0x1069,0,30dpd_6mob,2023-07-01
2,CUS_0x114a_2023_01_01,CUS_0x114a,0,30dpd_6mob,2023-07-01
3,CUS_0x1184_2023_01_01,CUS_0x1184,0,30dpd_6mob,2023-07-01
4,CUS_0x1297_2023_01_01,CUS_0x1297,1,30dpd_6mob,2023-07-01


In [32]:
gold_label_df.groupBy("label", "label_def").count().show()

gold_feature_df.select(
    "Customer_ID",
    "snapshot_date",
    "debt_to_income_ratio",
    "emi_to_salary_ratio",
    "high_credit_utilization_flag"
).show(10)

+-----+----------+-----+
|label| label_def|count|
+-----+----------+-----+
|    0|30dpd_6mob| 6383|
|    1|30dpd_6mob| 2591|
+-----+----------+-----+

+-----------+-------------+--------------------+--------------------+----------------------------+
|Customer_ID|snapshot_date|debt_to_income_ratio| emi_to_salary_ratio|high_credit_utilization_flag|
+-----------+-------------+--------------------+--------------------+----------------------------+
| CUS_0x47a5|   2023-01-01|0.050223153504446776|0.021862522013473804|                           0|
| CUS_0x51c9|   2023-01-01|0.009708455480298816|                 0.0|                           0|
| CUS_0xb0c4|   2023-01-01| 0.16582633812200587| 0.02820225693529963|                           0|
| CUS_0xc63b|   2023-01-01|0.023721193233179037|                 0.0|                           0|
| CUS_0x4c71|   2023-01-01|0.013354632384071918|0.015207769116710425|                           0|
| CUS_0xc38a|   2023-01-01|0.013689406384097122|0.0188432

In [33]:
gold_feature_df.groupBy("snapshot_date").count().orderBy("snapshot_date").show(50)

gold_label_df.groupBy("snapshot_date").count().orderBy("snapshot_date").show(50)

+-------------+-----+
|snapshot_date|count|
+-------------+-----+
|   2023-01-01|  530|
|   2023-02-01|  501|
|   2023-03-01|  506|
|   2023-04-01|  510|
|   2023-05-01|  521|
|   2023-06-01|  517|
|   2023-07-01|  471|
|   2023-08-01|  481|
|   2023-09-01|  454|
|   2023-10-01|  487|
|   2023-11-01|  491|
|   2023-12-01|  489|
|   2024-01-01|  485|
|   2024-02-01|  518|
|   2024-03-01|  511|
|   2024-04-01|  513|
|   2024-05-01|  491|
|   2024-06-01|  498|
|   2024-07-01|  505|
|   2024-08-01|  543|
|   2024-09-01|  493|
|   2024-10-01|  456|
|   2024-11-01|  488|
|   2024-12-01|  515|
+-------------+-----+

+-------------+-----+
|snapshot_date|count|
+-------------+-----+
|   2023-07-01|  530|
|   2023-08-01|  501|
|   2023-09-01|  506|
|   2023-10-01|  510|
|   2023-11-01|  521|
|   2023-12-01|  517|
|   2024-01-01|  471|
|   2024-02-01|  481|
|   2024-03-01|  454|
|   2024-04-01|  487|
|   2024-05-01|  491|
|   2024-06-01|  489|
|   2024-07-01|  485|
|   2024-08-01|  518|
|   2024-

In [34]:
gold_feature_df = spark.read.parquet("datamart/gold/feature_store/*")
gold_label_df = spark.read.parquet("datamart/gold/label_store/*")

join_df = gold_feature_df.join(
    gold_label_df,
    on=["Customer_ID", "snapshot_date"],
    how="inner"
)

print("Feature rows:", gold_feature_df.count())
print("Label rows:", gold_label_df.count())
print("Joined rows:", join_df.count())

join_df.show(5)

Feature rows: 11974
Label rows: 8974
Joined rows: 0
+-----------+-------------+----+---+---+----------+-------------+---------------------+-----------------+---------------+-------------+-----------+------------+-------------------+----------------------+--------------------+--------------------+----------+----------------+------------------------+------------------+---------------------+-------------------+-----------------------+-----------------+---------------+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+--------------------+-------------------+----------------------------+-------+-----+---------+
|Customer_ID|snapshot_date|Name|Age|SSN|Occupation|Annual_Income|Monthly_Inhand_Salary|Num_Bank_Accounts|Num_Credit_Card|Interest_Rate|Num_of_Loan|Type_of_Loan|Delay_from_due_date|Num_of_Delayed_Payment|Changed_Credit_Limit|Num_Credit_Inquiries|Credit_Mix|Outstanding_Debt|Credit_Utilization_Ratio|Credit_History_Age|Payment_

In [35]:
# Check if Customer_ID overlaps
customer_overlap = gold_feature_df.select("Customer_ID").distinct().join(
    gold_label_df.select("Customer_ID").distinct(),
    on="Customer_ID",
    how="inner"
)

print("Common customers:", customer_overlap.count())

# Check if snapshot_date overlaps
date_overlap = gold_feature_df.select("snapshot_date").distinct().join(
    gold_label_df.select("snapshot_date").distinct(),
    on="snapshot_date",
    how="inner"
)

print("Common snapshot dates:", date_overlap.count())

Common customers: 8974
Common snapshot dates: 18


In [36]:
debug_df = gold_feature_df.alias("f").join(
    gold_label_df.alias("l"),
    on="Customer_ID",
    how="inner"
).select(
    col("Customer_ID"),
    col("f.snapshot_date").alias("feature_snapshot_date"),
    col("l.snapshot_date").alias("label_snapshot_date")
)

debug_df.show(20)

+-----------+---------------------+-------------------+
|Customer_ID|feature_snapshot_date|label_snapshot_date|
+-----------+---------------------+-------------------+
| CUS_0x47a5|           2023-01-01|         2023-07-01|
| CUS_0x51c9|           2023-01-01|         2023-07-01|
| CUS_0xb0c4|           2023-01-01|         2023-07-01|
| CUS_0xc63b|           2023-01-01|         2023-07-01|
| CUS_0x4c71|           2023-01-01|         2023-07-01|
| CUS_0xc38a|           2023-01-01|         2023-07-01|
| CUS_0x6752|           2023-01-01|         2023-07-01|
| CUS_0xb745|           2023-01-01|         2023-07-01|
| CUS_0x1325|           2023-01-01|         2023-07-01|
| CUS_0x3231|           2023-01-01|         2023-07-01|
| CUS_0x325f|           2023-01-01|         2023-07-01|
| CUS_0x8065|           2023-01-01|         2023-07-01|
| CUS_0xa488|           2023-01-01|         2023-07-01|
| CUS_0x3d6c|           2023-01-01|         2023-07-01|
| CUS_0x63d8|           2023-01-01|         2023